**`¿qué atributos describen una fase?`**    
el modelo va aprender sobre los atributos de la fase.  

**`¿cómo se agrupan esas fases dentro de familias?`**     
no he de trabajar con fase - máquina sino más bien familias de fases - máquinas.   

**`revisar los tiempos reales`**      
puede haber mucho ruido y he de controlarlo.   

**`construir la tabla de conteo`**
conteo de familias de fases x máquinas

atributos → familias → tabla de conteos familia × máquina → lectura de dónde hay decisión

In [654]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [655]:
ruta_FASE = os.path.join("res\\raw", "CPRMOTask.csv")
print(ruta_FASE) 

res\raw\CPRMOTask.csv


In [656]:
columns_FASE = ["IDMOTask", "CodCompany", "IDManufacturingOrder", "CodMOTask", "Description", "Notes", "Order", "PreparationTime", "ExecutionTime", "RepetitionTime", "TimeUnit", "ProductEntry", "MaterialExit", "DelimitationType", "DelimitationDate", "AllowFraccionate", "MinimumFraction", "IDUsualTask", "TaskType", "IDMOTaskParent", "RepetitionLot", "External", "IDArticleExternal", "Quantity", "IDSupplier", "IDTaskType", "RepetitionTimeReal", "PlannedStartDate", "PlannedEndDate", "Group", "IDTask", "PercentProgress", "SeriesRequired", "QualityStatus", "InspectedDate", "Priority", "OptimizationFeature", "Fixed", "WithoutTimetable", "OccupancyType", "OccupancyHours", "OccupancyPercent", "ManualEndDate", "ManualStartDate", "Blocked", "DelimitationObligatory", "IDBudgetCategory", "ResourcePlanType", "SplitNumber", "ManualPlanning", "IDBudgetEmployee", "IDBudgetMachine", "IDBudgetTool", "IDBudgetMaterial", "IDBudgetOtherCost", "RowTimestamp", "WorkUnits", "LastModifiedBy", "IDProductFormula", "DateReplanify", "LotsOrder", "PrintLabels", "AutoExecutionImputation", "IDSiteMaterial", "IDWarehouseMaterial", "GroupingNumber", "CreationTimestamp", "RowCreatedBy", "BlockedMaterial", "BlockedCost", "ImputationCostUpdated", "Canceled", "RealEndDate", "IDCheckList", "NotOverlapCode", "TiedCode", "GroupedManufacturing", "ReleaseKeptResources", "DelayMargin", "BlockedChangeMaterial", "IDHourType", "Color", "SetToQtyConversionFactor", "AdjustTaskToDay", "PercentProgressType", "StatusTaskMES"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)          

df_FASE = pd.read_csv(ruta_FASE, sep=";", header=None, names=columns_FASE)

C:\Users\jaume\AppData\Local\Temp\ipykernel_21096\2516382180.py:6: DtypeWarning: Columns (0: IDSiteMaterial, 1: IDWarehouseMaterial, 2: GroupingNumber, 3: IDHourType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_FASE = pd.read_csv(ruta_FASE, sep=";", header=None, names=columns_FASE)


In [657]:
display(df_FASE.sample(1))

,IDMOTask,CodCompany,IDManufacturingOrder,CodMOTask,Description,Notes,Order,PreparationTime,ExecutionTime,RepetitionTime,TimeUnit,ProductEntry,MaterialExit,DelimitationType,DelimitationDate,AllowFraccionate,MinimumFraction,IDUsualTask,TaskType,IDMOTaskParent,RepetitionLot,External,IDArticleExternal,Quantity,IDSupplier,IDTaskType,RepetitionTimeReal,PlannedStartDate,PlannedEndDate,Group,IDTask,PercentProgress,SeriesRequired,QualityStatus,InspectedDate,Priority,OptimizationFeature,Fixed,WithoutTimetable,OccupancyType,OccupancyHours,OccupancyPercent,ManualEndDate,ManualStartDate,Blocked,DelimitationObligatory,IDBudgetCategory,ResourcePlanType,SplitNumber,ManualPlanning,IDBudgetEmployee,IDBudgetMachine,IDBudgetTool,IDBudgetMaterial,IDBudgetOtherCost,RowTimestamp,WorkUnits,LastModifiedBy,IDProductFormula,DateReplanify,LotsOrder,PrintLabels,AutoExecutionImputation,IDSiteMaterial,IDWarehouseMaterial,GroupingNumber,CreationTimestamp,RowCreatedBy,BlockedMaterial,BlockedCost,ImputationCostUpdated,Canceled,RealEndDate,IDCheckList,NotOverlapCode,TiedCode,GroupedManufacturing,ReleaseKeptResources,DelayMargin,BlockedChangeMaterial,IDHourType,Color,SetToQtyConversionFactor,AdjustTaskToDay,PercentProgressType,StatusTaskMES
70938,9c5f285b-0db2-445e-a6ba-9a4c172f63a6,100,b2f859dc-68cc-4de3-ae7d-76113b1f14b9,06,Mecanització al H15 C1+C2,NaN,20,0,0,0,1,0,0,1,NaN,0,0,64dbdeef-a81c-42f8-83fa-ac6e953af375,1,NaN,0.0,0,NaN,10.0,NaN,NaN,0,2025-07-29 00:00:00.000,2025-07-29 08:00:00.000,NaN,NaN,100.0,0,1,NaN,0,NaN,0,0,1,24.0,100.0,2025-07-29 00:00:00.000,NaN,0,0,NaN,1,1,0,NaN,NaN,NaN,NaN,NaN,2025-08-26 06:49:04.810,0.0,OLANET,NaN,2025-07-29 00:00:00.000,0,0,0,NaN,NaN,NaN,2025-07-29 07:32:15.410,HHAM,0,0,1,0,2025-08-26 00:00:00.000,NaN,NaN,NaN,0,0,0.0,0,NaN,0.0,1.0,0,1,4


In [658]:
display(df_FASE.shape)

(116055, 86)

## Limpiar fases Externas o Canceladas
Todas estas fases no sirven para entrenar al modelo.

In [659]:
def filtrar_fases_no_validas(df):
    """Conserva solo las fases que NO son externas, NI canceladas.
    Descarta cualquier instancia con un 1 en External o Canceled.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    n_inicial = df.shape[0]

    # Conservar filas donde las tres valen 0 (o NULL tratado como 0).
    # fillna(0): si hubiera NULLs, los tratamos como "no marcado" = válido.
    mask_validas = (
        (df['External'].fillna(0) == 0) &
        (df['Canceled'].fillna(0) == 0)
    )

    df_out = df[mask_validas].copy()

    # Desglose de cuántas se van por cada motivo (informativo; pueden solaparse)
    n_ext = (df['External'].fillna(0) == 1).sum()
    n_can = (df['Canceled'].fillna(0) == 1).sum()

    print(f"Partida: {n_inicial} fases")
    print(f"  Externas:  {n_ext}") # External=1
    print(f"  Canceladas: {n_can}") # Canceled=1
    print(f"Conservadas: {df_out.shape[0]} fases "
          f"(descartadas {n_inicial - df_out.shape[0]})")

    return df_out

In [660]:
df_FASE = filtrar_fases_no_validas(df_FASE)

Partida: 116055 fases
  Externas:  8501
  Canceladas: 0
Conservadas: 107554 fases (descartadas 8501)


Puede ser que haya fases Externas infiltradas, voy hacer uns inspección en base al nombre de la fase

In [661]:
posibles_externas = "ext|subcon"

df_infiltradas = df_FASE[df_FASE["Description"].str.contains(posibles_externas, case=False, na=False)]

print(f"Número de fases externas infiltradas (no marcadas como externas siendolo): {df_infiltradas.shape[0]}" )
#display(df_infiltradas.sample(df_infiltradas.shape[0]))

Número de fases externas infiltradas (no marcadas como externas siendolo): 12


Estas fases externas infiltradas no las quiero, asi que las elimino del dataset con el que trabajaré.

In [662]:
print(f"Fases totales de eliminar infiltradas antes: {df_FASE.shape[0]}")

df_FASE = df_FASE[~df_FASE["Description"].str.contains(posibles_externas, case=False, na=False)]

print(f"Se eliminarán: {df_infiltradas.shape[0]}")
print(f"Fases totales después de eliminar infiltradas: {df_FASE.shape[0]}")

Fases totales de eliminar infiltradas antes: 107554
Se eliminarán: 12
Fases totales después de eliminar infiltradas: 107542


## Limpieza por fecha
Vamos a trabajar con instancias solo a partir del Junio 2025

> Esto lo quiero hacer por filtrado de OF

In [663]:
# def filtrar_desde_fecha(df, fecha_desde, columna='CreationTimestamp'):
#     """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
#     Convierte la columna a datetime por seguridad (sirve si viene como texto).
#     Devuelve un DataFrame nuevo."""

#     n_inicial = df.shape[0]
#     df_out = df.copy()

#     # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
#     # parseables se vuelven NaT (nulo temporal) en vez de reventar.
#     df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

#     # Avisar si alguna fecha no se pudo interpretar
#     n_nat = df_out[columna].isna().sum()
#     if n_nat:
#         print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
#               f"Quedan fuera del filtro >= {fecha_desde}.")

#     limite = pd.to_datetime(fecha_desde)
#     df_out = df_out[df_out[columna] >= limite].copy()

#     print(f"Partida: {n_inicial} fases")
#     print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
#           f"(descartadas {n_inicial - df_out.shape[0]})")
#     return df_out

In [664]:
# df_FASE = filtrar_desde_fecha(df_FASE, fecha_desde='2025-06-01')
# print(df_FASE.shape)

## Limpieza de atributos 

Tenemos 86 columnas, vamos hacer la selección de atributos:    

Atributos que sobran: [CodCompany, CodMOTask, Notes, Order, RepetitionTime, DelimitationType, DelimitationDate, AllowFraccionate, MinimumFraction, RepetitionLot, IDArticleExternal, RepetitionTimeReal, Group, SeriesRequired, QualityStatus, InspectedDate, Priority, OptimizationFeature, Fixed, WithoutTimetable, OccupancyType, DelimitationObligatory, IDBudgetCategory, ResourcePlanType, SplitNumber, ManualPlanning, IDBudgetEmployee, IDBudgetMachine, IDBudgetTool, IDBudgetMaterial, IDBudgetOtherCost, RowTimestamp, LastModifiedBy, WorkUnits, IDProductFormula, LotsOrder, PrintLabels, AutoExecutionImputation, IDSiteMaterial, IDWarehouseMaterial, GroupingNumber, CreationTimestamp, RowCreatedBy, BlockedMaterial, BlockedCost, ImputationCostUpdated, IDCheckList, NotOverlapCode, TiedCode, GroupedManufacturing, ReleaseKeptResources, DelayMargin, BlockedChangeMaterial, IDHourType, Color, SetToQtyConversionFactor, AdjustTaskToDay, PercentProgressType, StatusTaskMES, Blocked]     

Atributos que necesito saber que posibles valores hay, y si son todos iguales, puedo prescindir de estos para el df: [WorkUnits, TimeUnit, OccupancyHours, OccupancyPercent]    

In [665]:
def inspeccionar_candidatos(df):
    """Inspecciona columnas candidatas a ser prescindibles.
    Para cada una muestra cuántos valores distintos tiene y su distribución.
    Marca cuáles son constantes (un solo valor) y por tanto prescindibles.
    NO se elimina nada, solo carácter informativo."""

    columnas = ['TimeUnit', 'OccupancyHours', 'OccupancyPercent']

    constantes = []
    con_varianza = []

    for col in columnas:
        if col not in df.columns:
            print(f"[{col}] no está en el DataFrame\n")
            continue

        n = df[col].nunique(dropna=False)
        print(f"====== {col} ======")
        print(f"Valores distintos (incluyendo NaN): {n}")
        print(df[col].value_counts(dropna=False).head(10).to_string())

        if n <= 1:
            constantes.append(col)
        else:
            con_varianza.append(col)

    print(f"Constantes: {constantes}")
    print(f"Con varianza:  {con_varianza}")

    return constantes, con_varianza

In [666]:
constantes, con_varianza = inspeccionar_candidatos(df_FASE)

====== TimeUnit ======
Valores distintos (incluyendo NaN): 3
TimeUnit
1    103958
2      3563
3        21
====== OccupancyHours ======
Valores distintos (incluyendo NaN): 1
OccupancyHours
24.0    107542
====== OccupancyPercent ======
Valores distintos (incluyendo NaN): 1
OccupancyPercent
100.0    107542
Constantes: ['OccupancyHours', 'OccupancyPercent']
Con varianza:  ['TimeUnit']


El único atributo que si tiene distintos valores es TimeUnit: 
Esta asigna valores de unidad temporal a cada Fase, es decir, con que unidad de tiempo se miden los valores de tiempo (preparación y ejecución)
- 1 = Minutos
- 2 = Horas
- 3 = Dias
- 4 = Semanas
- 5 = Meses

In [667]:
# normalizo los tiempos teóricos a minutos según TimeUnit
factor_min = {1: 1, 2: 60, 3: 60 * 24}   # ojo con el factor de días

df_FASE["PreparationTime"] = df_FASE["PreparationTime"] * df_FASE["TimeUnit"].map(factor_min)
df_FASE["ExecutionTime"] = df_FASE["ExecutionTime"] * df_FASE["TimeUnit"].map(factor_min)

Tanto el tiempo de preparación como el tiempo de ejecución debo pasarlo a minutos para aquellos que tengan el valor `2` en la columna `TimeUnit `.  
        
Inspeccionado esas fases y hablando con oficina técnica vemos que realmente el error esta en la unidad especificada, pero los valores numéricos del tiempo si sóestan dados en minutos, siendo entonces correctos, con lo que no debemos modificar esos tiempos. Entonces podemos prescindir de la columna `TimeUnit` y tomar los tiempos como minitos.   

Mantengo TimeUnit porque he visto problemas en la Viabilidad de los datos.

In [668]:
def eliminar_columnas(df):
    """Elimina de df_FASES las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original)."""

    cols_sobran = [
        'CodMOTask','Notes','Order','RepetitionTime','DelimitationType',
        'DelimitationDate','AllowFraccionate','MinimumFraction','RepetitionLot', 'CodCompany',
        'IDArticleExternal','RepetitionTimeReal','Group','SeriesRequired','QualityStatus',
        'InspectedDate','Priority','OptimizationFeature','Fixed','WithoutTimetable',
        'OccupancyType','DelimitationObligatory','IDBudgetCategory','ResourcePlanType',
        'SplitNumber','ManualPlanning','IDBudgetEmployee','IDBudgetMachine','IDBudgetTool',
        'IDBudgetMaterial','IDBudgetOtherCost','RowTimestamp','LastModifiedBy', 'WorkUnits',
        'IDProductFormula','LotsOrder','PrintLabels','AutoExecutionImputation', 'External',
        'IDSiteMaterial','IDWarehouseMaterial','GroupingNumber','CreationTimestamp',
        'RowCreatedBy','BlockedMaterial','BlockedCost','ImputationCostUpdated', 'IDMOTaskParent',
        'IDCheckList','NotOverlapCode','TiedCode','GroupedManufacturing','Blocked',
        'ReleaseKeptResources','DelayMargin','BlockedChangeMaterial','IDHourType','Color',
        'SetToQtyConversionFactor','AdjustTaskToDay','PercentProgressType','StatusTaskMES',
        'OccupancyHours', 'OccupancyPercent', 'TaskType', 'IDTask', 'PercentProgress', 'Canceled',
        'IDSupplier', 'IDTaskType', 'DateReplanify', 'PlannedStartDate', 'PlannedEndDate', 'ManualEndDate'
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_FASE]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_FASE.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [669]:
df_FASE = eliminar_columnas(df_FASE)

display(df_FASE.head(1))

Eliminadas 74 columnas. Quedan 12 columnas, 107542 filas.


,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate
0,00003219-f775-4bea-9e04-dbafb6e8fd80,7de89fb9-e8db-48bf-b822-9c52431c44a1,Marcatges,1,1,1,0,0,134bdf39-e99c-4be8-af7a-36b7899c152d,1.0,2026-01-19 15:58:00.000,2026-01-23 00:00:00.000


Perfecto, hasta aqui tengo df:
- sin las fases externas o canceladas
- sin fases con tiempos teoricos a 0
- instancias hechas en un intervalo de tiempo correcto
- sin los atributos que no aportan nada

## Eliminar fases que su OF debe finalizarse en Agosto
De esta forma solo contaremos con OF termindas, teniendo estas todo su tiempo imputado, siendo así los tiempos reales lícitos.  

In [670]:
df_OF = pd.read_parquet("res//procesed/df_OF.parquet")
display(df_OF.head(1))
print(f"Total de OFs: {df_OF.shape[0]} con {df_OF.shape[1]} atributos")

,IDManufacturingOrder,CodCompany,CodManufacturingOrder,IDArticle,Description,IDStructureType,Quantity,IDMOSituation,RealEndDate,IDProject,QuantityBaseFormula,IDUnitBaseFormula
0,000744d2-5b9e-40be-9126-9a8a66954f33,100,OF2502249,921ac292-f882-47f8-ae0b-ce82c4f1a3c8,CONDUCTEUR 103127410,d3bd7234-a0af-4c35-b658-dcd8ac943c67,7.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2025-04-17 00:00:00.000,NaN,7.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb


Total de OFs: 17284 con 12 atributos


Entonces si una fase (`df_FASE`) no pertenece a ninguna OF (`df_OF`) debemos eliminarla de `df_FASE`.    
La PK de `df_FASE` es una FK en `df_FASE`.

In [671]:
print(f"Total de Fases antes de filtrar por OFs válidas: {df_FASE.shape[0]}")

df_FASE = df_FASE[df_FASE["IDManufacturingOrder"].isin(df_OF["IDManufacturingOrder"])]
print(f"Total de Fases después de filtrar por OFs válidas: {df_FASE.shape[0]}")

Total de Fases antes de filtrar por OFs válidas: 107542
Total de Fases después de filtrar por OFs válidas: 102800


Ahora vamos a prescindir de todas las fases que no pertenezcan a ninguna una OF.

El dataset OF la columna `IDManufacturingOrder` es la PK que identifica cada instancia del dataset de forma única.     
El dataset de Fases la columna `IDManufacturingOrder` es la FK que identifica a que OF pertenece la fase.

Estas fases eliminadas pertenecen a OFs las cuales no podemos garantizar que esten terminadas, por lo tanto no podemos garantizar que los tiempos reales capturados estén completos, siendo lo más sensato eliminarlas.

In [672]:
display(df_FASE.head(1))
print(df_FASE.shape)

,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate
0,00003219-f775-4bea-9e04-dbafb6e8fd80,7de89fb9-e8db-48bf-b822-9c52431c44a1,Marcatges,1,1,1,0,0,134bdf39-e99c-4be8-af7a-36b7899c152d,1.0,2026-01-19 15:58:00.000,2026-01-23 00:00:00.000


(102800, 12)


---

## Unión con imputaciones en MES
Voy a unir ahora `df_FASE` con mis imputaciones limpias de `df_fases_MES`  

### Añadir ID de MES
La tabla `df_FASE` esta estraida de ERP, y la `df_fases_MES` de sistema MES, por lo tanto los ID de dce las FASES no coinciden entre ambas tablas.    
Existe una BBDD existe una tabla de mapeo entre endidades de ERP y las de MES, ahi en cada instancia podemos encontrar el ID de cada sistema y su homologo. Voy a tomar el ID de MES y añadirlo a la tabla `df_FASSE`. Esto es necesario, porque sino no podre relacionar las imputaciones a la tabla. 

In [673]:
ruta_maping = os.path.join("res\\raw", "maping_fase.csv")
print(ruta_FASE) 


res\raw\CPRMOTask.csv


In [674]:
columns_maping = ["SmERPEntityMappingUid",	"ERPEntityType",	"ERPEntityId",	"MESEntityType",	"MESEntityId",	"bajaInd",	"FechaUltimaActualizacionDte",	"Planta",	"modificadoPor_usuario_guid",	"FechaCreacionDte"]     

df_maping_fase = pd.read_csv(ruta_maping, sep=";", header=None, names=columns_maping)

In [675]:
display(df_maping_fase.head(1))

,SmERPEntityMappingUid,ERPEntityType,ERPEntityId,MESEntityType,MESEntityId,bajaInd,FechaUltimaActualizacionDte,Planta,modificadoPor_usuario_guid,FechaCreacionDte
0,D858848B-E9B7-4146-84E9-00005456200F,Basoa.Services.Manufacturing.Entities.MOTask,91c75951-3e60-4c7d-bbeb-432326bf3328,SmFase,f6d2d696-44e8-492e-80be-70dc47c7089c,0,2026-01-23 14:19:17.000,DESARROLLO,NaN,2026-01-23 14:19:20.777


In [676]:
df_maping_fase = df_maping_fase.drop(["SmERPEntityMappingUid", "MESEntityType", "ERPEntityType", "bajaInd", "FechaUltimaActualizacionDte", "Planta", "modificadoPor_usuario_guid", "FechaCreacionDte"], axis=1)
display(df_maping_fase.head(1))

,ERPEntityId,MESEntityId
0,91c75951-3e60-4c7d-bbeb-432326bf3328,f6d2d696-44e8-492e-80be-70dc47c7089c


In [677]:
# Uno con merge
df_FASE = df_FASE.merge(df_maping_fase, left_on ="IDMOTask", right_on="ERPEntityId",  how="left")

In [678]:
print(f"Fases sin correlativa en MES: {df_FASE["MESEntityId"].isna().sum()}")
df_FASE = df_FASE.drop(columns="ERPEntityId") # la elimina porque esta columna queda duplicada
display(df_FASE.head(1))

Fases sin correlativa en MES: 0


,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate,MESEntityId
0,00003219-f775-4bea-9e04-dbafb6e8fd80,7de89fb9-e8db-48bf-b822-9c52431c44a1,Marcatges,1,1,1,0,0,134bdf39-e99c-4be8-af7a-36b7899c152d,1.0,2026-01-19 15:58:00.000,2026-01-23 00:00:00.000,5b0488ec-78f9-4dfb-8bf8-08396bdf4e69


In [679]:
df_fases_MES = pd.read_parquet("res//procesed/df_fases_MES.parquet")

display(df_fases_MES.head(1))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
0,00013C31-3895-4370-BB3E-4255371E8D8F,TN09,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,41.76667,41.76667,0.0,50.0,2,0bab3b63-5d31-4de3-85e2-89dc5a3e9bd5,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,27451f15-090c-42a6-82d6-98b7451db3db,TPU,Torns Puma (Taller COURES)


In [680]:
# Los id en MES estan con minúsculas y en ERP en mayusculas
df_fases_MES["SmFaseUid"] = df_fases_MES["SmFaseUid"].str.lower().str.strip()
df_FASE["MESEntityId"] = df_FASE["MESEntityId"].str.lower().str.strip()

In [681]:
df_FASE_final = df_fases_MES.merge(df_FASE, left_on="SmFaseUid", right_on="MESEntityId", how="left")
print(f"Total de fases después del join: {len(df_FASE_final)}")
print(f"Total de fases a decartar: {len(df_FASE)-len(df_FASE_final)}")

display(df_FASE_final.head(1))

Total de fases después del join: 17389
Total de fases a decartar: 85411


,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo,IDMOTask,IDManufacturingOrder,Description,PreparationTime,ExecutionTime,TimeUnit,ProductEntry,MaterialExit,IDUsualTask,Quantity,ManualStartDate,RealEndDate,MESEntityId
0,00013c31-3895-4370-bb3e-4255371e8d8f,TN09,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,41.76667,41.76667,0.0,50.0,2,0bab3b63-5d31-4de3-85e2-89dc5a3e9bd5,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,27451f15-090c-42a6-82d6-98b7451db3db,TPU,Torns Puma (Taller COURES),ee76d685-a4cc-47d4-b418-6e33d2ae2921,26a3fecc-120e-49bf-9e77-720d35355e3d,Mecanitzat al Torn Puma Petit,6.0,15.0,1.0,0.0,0.0,29783995-2f56-48be-9664-461e6f0b402a,3.0,2025-10-13 12:55:00.000,2025-10-02 00:00:00.000,00013c31-3895-4370-bb3e-4255371e8d8f


In [682]:
print(f"Total fases df_FASE_final: {len(df_FASE_final)}")
print(f"Fases SIN match en ERP (MESEntityId NaN): {df_FASE_final['MESEntityId'].isna().sum()} - son fases con imputaciones en OF's fuera del intervalo de estudio.")
print(f"Fases CON match: {df_FASE_final['MESEntityId'].notna().sum()}")

Total fases df_FASE_final: 17389
Fases SIN match en ERP (MESEntityId NaN): 555 - son fases con imputaciones en OF's fuera del intervalo de estudio.
Fases CON match: 16834


In [683]:
# 1. descarto las que no tienen match ERP (las 555)
antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[df_FASE_final["MESEntityId"].notna()]
print(f"Descartadas sin match ERP: {antes - len(df_FASE_final)} | quedan: {len(df_FASE_final)}")

Descartadas sin match ERP: 555 | quedan: 16834


Tras cruzar las fases con tiempo real (MES) y su tiempo teórico correspondiente (ERP), se detectaron fases sin correspondencia en el sistema teórico. Estas pertenecen a órdenes de fabricación que quedaron fuera del rango temporal de finalización del estudio.

En la depuración previa, las órdenes de fabricación se filtraron por su fecha de finalización, ya que en una orden no finalizada dentro del intervalo no puede garantizarse que sus imputaciones de tiempo estén completas, y el tiempo real registrado podría no reflejar el trabajo total de la fase. Al descartar dichas órdenes del ERP, sus fases quedaron sin tiempo teórico asociado.

Dado que estas fases carecen de tiempo teórico y que el ratio real/teórico constituye la variable objetivo, se descartan del análisis. Esta exclusión es coherente con el filtrado temporal previamente adoptado y garantiza que el conjunto final se componga únicamente de fases con imputaciones completas y ambos tiempos disponibles.

> `df_FASES_final` tiene todas las fases de aquellas maquinas con las que trabajaremos el modelo.

## Analizar fases sin tiempos teóricos
Todas las fases que no tengan tiempos en los atributos `PreparationTime`  o `ExecutionTime` son problemáticas para el dataset, esas instancias no nos serán útiles.

In [684]:
print(f"Instancias totales: {df_FASE_final.shape[0]}")
df_FASE_con_TIEMPOS = df_FASE_final[(df_FASE_final["PreparationTime"] != 0) | (df_FASE_final["ExecutionTime"] != 0)]

print(f"Número de instancias con tiempos teóricos: {df_FASE_con_TIEMPOS.shape[0]}")
print(f"Número de instancias sin tiempo teórico (descartar): {df_FASE_final.shape[0] - df_FASE_con_TIEMPOS.shape[0]}")

Instancias totales: 16834
Número de instancias con tiempos teóricos: 15383
Número de instancias sin tiempo teórico (descartar): 1451


Cuando se creó la OF, para esas fases no se estimó (o no se conocía) el tiempo teórico, y quedó en 0.

In [685]:
teorico_cero = df_FASE_final[
    (df_FASE_final["PreparationTime"] == 0) & (df_FASE_final["ExecutionTime"] == 0)
]
print(f"Fases con teórico 0: {len(teorico_cero)}")
print(teorico_cero["MinutosReal_total"].describe())

Fases con teórico 0: 1451
count    1451.000000
mean      169.723731
std       138.993352
min         0.166670
25%        66.358330
50%       132.233340
75%       239.274995
max       860.850000
Name: MinutosReal_total, dtype: float64


In [686]:
print(teorico_cero["NombreGrupo"].value_counts())

NombreGrupo
Centre de Mecanitzat 5 Eixos petites        459
Torns Control Numèric                       390
Centres de Mecanitzat Vertical COURES       277
Centres de Mecanitzat Vertical ALUMINIS     146
Centres de Mecanitzat Horitzontal COURES    123
Premses Conformats                           23
Torns Puma (Taller COURES)                   21
Màquines Secció Flexibles                     8
Màquines de Taladrat Profund                  4
Name: count, dtype: int64


In [687]:
antes = len(df_FASE_final)
df_FASE_final = df_FASE_con_TIEMPOS
print(f"Descartadas (teórico 0): {antes - len(df_FASE_final)} | quedan: {len(df_FASE_final)}")

Descartadas (teórico 0): 1451 | quedan: 15383


Tenemos solo las fases que pertenecen a maquinas productivas. Vamos a inspeccionar todas estas fases para ver como se agrupan.

In [688]:
#print(df_FASE_final["smFaseNme"].value_counts())
print("")
print(f"Total de tipos de fases en base su nombre: {len(set(df_FASE_final["smFaseNme"]))}")

#print(df_FASE_final["Description"].value_counts())
print("")
print(f"Total de tipos de fases en base su nombre: {len(set(df_FASE_final["Description"]))}")


Total de tipos de fases en base su nombre: 50

Total de tipos de fases en base su nombre: 151


Hablando con Oficina Técnica llegamos a un listado de fases a descartar, ya que son fases con condiciones fuera de mi población de estudio.

In [689]:
tipos_no_productivos = ["MAR", "PM", "AKS", "AKS-ALU", "ASO", "TRT", "TRT1", 
                        "OFI", "OFI-ALU", "OFI-TCN", "SOL", "TCN-EXT"]

antes = len(df_FASE_final)
df_FASE_final = df_FASE_final[~df_FASE_final["smFaseNme"].isin(tipos_no_productivos)]
print(f"Eliminadas por tipo no productivo: {antes - len(df_FASE_final)}")
print(f"Fases restantes: {len(df_FASE_final)}")

Eliminadas por tipo no productivo: 253
Fases restantes: 15130


Estas fases cuenta con imputaciones correctas, pero voy a tener que eliminarlas para mi proyecto ya que necesito los tiempos teoricos. 

In [690]:
df_FASE_final.to_parquet("res//procesed/df_FASE_final.parquet")